#Credit Scoring System

Mount Drive and Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install mlflow scikit-learn pandas numpy joblib gradio

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Import Necessary Dependencies and Setup Environment


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import gradio as gr
import mlflow
import mlflow.sklearn
import joblib

Data Processing Pipeline

In [ ]:
# 2. LOAD DATA
# (Make sure this path is still correct for your Drive)
df = pd.read_csv('/content/drive/MyDrive/Dataset/Credit Score Classification Dataset.csv')

# 3. SEPARATE FEATURES
# Assume target is 'Credit Score'
X = df.drop('Credit Score', axis=1)
y = df['Credit Score']

# Split data to calculate accuracy for MLflow
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. DEFINE PREPROCESSING & PIPELINE
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('classifier', RandomForestClassifier())])

Training and Tracking of Models

In [ ]:
mlflow.set_experiment("credit-score-sagemaker-v1")
# 5. TRAIN & LOG WITH MLFLOW
# We wrap training in a run to track it
with mlflow.start_run() as run:
    print(f"Starting MLflow Run ID: {run.info.run_id}")

    # Train
    pipeline.fit(X_train, y_train)

    # Calculate Metric
    accuracy = pipeline.score(X_test, y_test)
    print(f"✅ Model Accuracy: {accuracy:.4f}")

    # Log Metric & Model
    mlflow.log_metric("accuracy", accuracy)
    mlflow.sklearn.log_model(pipeline, "model")
    print("✅ Model saved to MLflow artifacts.")

# 6. DEFINE PREDICTION FUNCTION (Safe Version)
def predict_credit_score(age, gender, income, education, marital_status, children, home_ownership):
    # Create DataFrame with generic inputs
    input_df = pd.DataFrame([{
        'Age': age,
        'Gender': gender,
        'Income': income,
        'Education': education,
        'Marital Status': marital_status,
        'Number of Children': children,
        'Home Ownership': home_ownership
    }])

    # AUTO-FIX: Rename columns to match what X_train actually had
    # This prevents the "Column Mismatch" error from happening again
    if 'Annual Income' in X.columns:
        input_df.rename(columns={'Income': 'Annual Income'}, inplace=True)
    if 'Education Level' in X.columns:
        input_df.rename(columns={'Education': 'Education Level'}, inplace=True)

    try:
        # Predict
        prediction = pipeline.predict(input_df)[0]
        return str(prediction)
    except Exception as e:
        return f"Error: {str(e)}"



Starting MLflow Run ID: 2dbaeee6dccd4a3c8697428590fc3a8f


2025/12/16 16:22:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


✅ Model Accuracy: 0.9091
✅ Model saved to MLflow artifacts.


Save Model
```



In [ ]:
        # --- SAVE DIRECTLY TO GOOGLE DRIVE ---
save_path = '/content/drive/MyDrive/credit_score_model.pkl'
joblib.dump(pipeline, save_path)
print(f"✅ Model permanently saved to: {save_path}")

✅ Model permanently saved to: /content/drive/MyDrive/credit_score_model.pkl


In [ ]:
import tarfile
import os

# 1. Define the Entry Script (This tells AWS how to use your Pipeline)
script_content = """
import joblib
import os
import pandas as pd
import json

def model_fn(model_dir):
    # Load the full pipeline (includes the One-Hot Encoder!)
    model = joblib.load(os.path.join(model_dir, "credit_score_model.pkl"))
    return model

def input_fn(request_body, request_content_type):
    if request_content_type == 'application/json':
        data = json.loads(request_body)
        return pd.DataFrame([data])
    else:
        raise ValueError(f"Unsupported content type: {request_content_type}")

def predict_fn(input_data, model):
    return model.predict(input_data)

def output_fn(prediction, response_content_type):
    # Map the result back to readable text
    result_str = str(prediction[0])
    return json.dumps({'credit_score': result_str})
"""

# Write the script to a file
with open('sagemaker_entry.py', 'w') as f:
    f.write(script_content)
print("✅ sagemaker_entry.py created.")

# 2. Define where your model is (It's in Drive based on your notebook)
source_model_path = '/content/drive/MyDrive/credit_score_model.pkl'

# 3. Create the 'model-v2.tar.gz' package
output_filename = "model-v2.tar.gz"

with tarfile.open(output_filename, "w:gz") as tar:
    # Add the model file (renaming it to just 'credit_score_model.pkl' inside the zip)
    tar.add(source_model_path, arcname="credit_score_model.pkl")
    # Add the entry script
    tar.add("sagemaker_entry.py", arcname="sagemaker_entry.py")

print(f"✅ SUCCESS! {output_filename} is ready. Download it from the Files tab on the left!")

✅ sagemaker_entry.py created.
✅ SUCCESS! model-v2.tar.gz is ready. Download it from the Files tab on the left!


UI Launch

In [ ]:
# 7. LAUNCH UI
demo = gr.Interface(
    fn=predict_credit_score,
    inputs=[
        gr.Number(label="Age", value=30),
        gr.Dropdown(["Male", "Female"], label="Gender"),
        gr.Number(label="Income", value=50000),
        gr.Dropdown(["Bachelor's Degree", "Master's Degree", "High School Diploma", "Associate's Degree", "PhD"], label="Education"),
        gr.Dropdown(["Single", "Married"], label="Marital Status"),
        gr.Slider(0, 10, step=1, label="Number of Children"),
        gr.Dropdown(["Rented", "Owned"], label="Home Ownership")
    ],
    outputs="text",
    title="Credit Score Predictor"
)

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://34783e959706ef75fa.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [1]:
# 1. Install specific versions
!pip install "numpy<2.0" scikit-learn==1.2.2 joblib pandas --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 47.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 9.7 MB/s eta 0:00:00
  Using cached scipy-1.16.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (62 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.2/509.2 kB 44.7 MB/s eta 0:00:00
Using cached scipy-1.16.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (35.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
import sklearn
print(f"Scikit-Learn Version: {sklearn.__version__}") # MUST say 1.2.2

Scikit-Learn Version: 1.2.2


In [4]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
import joblib
import tarfile
import os # Added for file operations

# Define the Entry Script (This tells AWS how to use your Pipeline)
script_content = """
import joblib
import os
import pandas as pd
import json

def model_fn(model_dir):
    # Load the full pipeline (includes the One-Hot Encoder!)
    model = joblib.load(os.path.join(model_dir, "credit_score_model.pkl"))
    return model

def input_fn(request_body, request_content_type):
    if request_content_type == 'application/json':
        data = json.loads(request_body)
        return pd.DataFrame([data])
    else:
        raise ValueError(f"Unsupported content type: {request_content_type}")

def predict_fn(input_data, model):
    return model.predict(input_data)

def output_fn(prediction, response_content_type):
    # Map the result back to readable text
    result_str = str(prediction[0])
    return json.dumps({'credit_score': result_str})
"""

# Write the script to a file
with open('sagemaker_entry.py', 'w') as f:
    f.write(script_content)
print("✅ sagemaker_entry.py created.")

# Train
df = pd.read_csv('/content/drive/MyDrive/Dataset/Credit Score Classification Dataset.csv')
X = df.drop('Credit Score', axis=1)
y = df['Credit Score']

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('classifier', RandomForestClassifier())])

pipeline.fit(X, y)
joblib.dump(pipeline, 'credit_score_model.pkl')

# Zip
with tarfile.open("model-v3-final.tar.gz", "w:gz") as tar:
    tar.add("credit_score_model.pkl")
    tar.add("sagemaker_entry.py")

print("✅ DONE! This file is now 1.2.2 compatible.")

✅ sagemaker_entry.py created.
✅ DONE! This file is now 1.2.2 compatible.
